In [0]:
df_encounters_bronze = spark.read.table("workspace.bronze.encounters")
df_encounters_bronze.printSchema()
df_encounters_bronze.show(10, truncate=False)

In [0]:
# Databricks notebook source
from pyspark.sql.functions import col, trim, to_date

df_encounters_bronze = spark.read.table("workspace.bronze.encounters")

df_encounters_silver = (
    df_encounters_bronze
    .filter(col("ID").isNotNull())
    .dropDuplicates(["ID"])
    .select(
        col("ID").alias("encounter_id"),
        col("PATIENT").alias("patient_id"),
        to_date(col("DATE"), "yyyy-MM-dd").alias("encounter_date"),
        col("CODE").alias("encounter_code"),
        trim(col("DESCRIPTION")).alias("encounter_description"),
        col("REASONCODE").alias("reason_code"),
        trim(col("REASONDESCRIPTION")).alias("reason_description"),
        col("ingested_at")
    )
)

(
    df_encounters_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.silver.encounters")
)

print(f"✅ Created workspace.silver.encounters with {df_encounters_silver.count()} clean rows!")